# 04. RFM Customer Segmentation

This notebook converts purchase behavior into actionable customer segments using Recency, Frequency, and Monetary value. Segment definitions are transparent and linked to lifecycle strategies.

## 1. Metric definitions

- **Recency:** days between the analysis snapshot date and the customer's latest purchase. Lower is better.
- **Frequency:** number of distinct completed orders. Higher is better.
- **Monetary:** total valid purchase revenue in GBP. Higher is better.

The snapshot date is one day after the final transaction date, which prevents the most recent customers from having zero-day recency.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter

pd.set_option('display.max_columns',None)
pd.set_option('display.float_format',lambda x:f'{x:,.2f}')
plt.style.use('seaborn-v0_8-whitegrid')
repo_root=Path.cwd()
if repo_root.name=='notebooks': repo_root=repo_root.parent
clean_path=repo_root/'data'/'processed'/'online_retail_clean.csv.gz'
image_dir=repo_root/'images'; output_dir=repo_root/'data'/'processed'
image_dir.mkdir(exist_ok=True)
df=pd.read_csv(clean_path,compression='gzip',parse_dates=['InvoiceDate'],dtype={'InvoiceNo':str,'StockCode':str,'CustomerID':str})
snapshot_date=df.InvoiceDate.max().normalize()+pd.Timedelta(days=1)
print(f'Analysis snapshot date: {snapshot_date.date()}')

## 2. Build the customer-level RFM table

In [ ]:
rfm=(df.groupby('CustomerID').agg(LastPurchase=('InvoiceDate','max'),Frequency=('InvoiceNo','nunique'),Monetary=('Revenue','sum')).reset_index())
rfm['Recency']=(snapshot_date-rfm.LastPurchase.dt.normalize()).dt.days
rfm=rfm[['CustomerID','Recency','Frequency','Monetary','LastPurchase']]
assert rfm.CustomerID.is_unique
assert (rfm[['Recency','Frequency','Monetary']]>0).all().all()
display(rfm.describe())
display(rfm.head())

## 3. Score customers by quartile

Recency is reverse-scored because fewer days indicate a more active customer. Frequency and Monetary are ranked before `qcut` so tied values—especially one-order customers—do not cause duplicate quantile edges.

In [ ]:
rfm['RScore']=pd.qcut(rfm.Recency.rank(method='first'),4,labels=[4,3,2,1]).astype(int)
rfm['FScore']=pd.qcut(rfm.Frequency.rank(method='first'),4,labels=[1,2,3,4]).astype(int)
rfm['MScore']=pd.qcut(rfm.Monetary.rank(method='first'),4,labels=[1,2,3,4]).astype(int)
rfm['RFMScore']=rfm.RScore.astype(str)+rfm.FScore.astype(str)+rfm.MScore.astype(str)
rfm['ValueScore']=rfm[['RScore','FScore','MScore']].mean(axis=1)
display(rfm[['RScore','FScore','MScore']].apply(pd.Series.value_counts).sort_index())

## 4. Assign lifecycle segments

Rules are applied in priority order so each customer belongs to exactly one segment. Monetary score is retained as a separate VIP flag and is included in the Champions definition.

In [ ]:
conditions=[
 (rfm.RScore==4)&(rfm.FScore==4)&(rfm.MScore>=3),
 (rfm.RScore>=3)&(rfm.FScore>=3),
 (rfm.RScore>=3)&(rfm.FScore<=2),
 (rfm.RScore<=2)&(rfm.FScore>=3),
 (rfm.RScore<=2)&(rfm.FScore<=2),
]
labels=['Champions','Loyal Customers','Potential Loyalists','At Risk','Hibernating']
rfm['Segment']=np.select(conditions,labels,default='Needs Attention')
rfm['VIPFlag']=np.where(rfm.MScore==4,'High Monetary Value','Standard Value')
assert rfm.Segment.notna().all()
assert rfm.groupby('CustomerID').size().eq(1).all()
print(rfm.Segment.value_counts())

## 5. Segment profile and revenue contribution

In [ ]:
segment_summary=(rfm.groupby('Segment').agg(Customers=('CustomerID','nunique'),Revenue=('Monetary','sum'),MedianRecency=('Recency','median'),AvgFrequency=('Frequency','mean'),AvgCustomerValue=('Monetary','mean')).reset_index())
segment_summary['CustomerShare']=segment_summary.Customers/segment_summary.Customers.sum()
segment_summary['RevenueShare']=segment_summary.Revenue/segment_summary.Revenue.sum()
segment_summary=segment_summary.sort_values('Revenue',ascending=False)
display(segment_summary)

fig,axes=plt.subplots(1,2,figsize=(14,6))
plot_data=segment_summary.sort_values('Customers')
axes[0].barh(plot_data.Segment,plot_data.Customers,color='#60A5FA')
axes[0].set_title('Customers by RFM Segment'); axes[0].set_xlabel('Customers')
axes[1].barh(plot_data.Segment,plot_data.RevenueShare,color='#1D4ED8')
axes[1].set_title('Revenue Share by RFM Segment'); axes[1].set_xlabel('Revenue share'); axes[1].xaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout(); plt.savefig(image_dir/'08_rfm_segment_profile.png',dpi=180,bbox_inches='tight'); plt.show()

## 6. Recency-frequency segment map

In [ ]:
rf_matrix=pd.crosstab(rfm.RScore,rfm.FScore).sort_index(ascending=False)
fig,ax=plt.subplots(figsize=(8,6)); im=ax.imshow(rf_matrix,cmap='Blues',aspect='auto')
ax.set_xticks(range(4),rf_matrix.columns); ax.set_yticks(range(4),rf_matrix.index)
ax.set_xlabel('Frequency score'); ax.set_ylabel('Recency score'); ax.set_title('Customer Count by Recency and Frequency Score')
for i in range(4):
 for j in range(4): ax.text(j,i,f'{rf_matrix.iloc[i,j]:,}',ha='center',va='center',color='white' if rf_matrix.iloc[i,j]>rf_matrix.values.max()*.55 else '#111827')
fig.colorbar(im,ax=ax,label='Customers'); plt.tight_layout(); plt.savefig(image_dir/'09_rfm_score_matrix.png',dpi=180,bbox_inches='tight'); plt.show()

## 7. Action strategy by segment

In [ ]:
strategy=pd.DataFrame([
 ['Champions','Protect and reward','VIP benefits, early access, referral incentives, premium bundles'],
 ['Loyal Customers','Increase advocacy and basket size','Loyalty rewards, cross-sell recommendations, milestone offers'],
 ['Potential Loyalists','Build the second-purchase habit','Personalized onboarding, replenishment reminders, second-order incentive'],
 ['At Risk','Prevent valuable customer loss','Win-back campaign, category-specific offer, service recovery survey'],
 ['Hibernating','Reactivate selectively','Low-cost reminder, time-limited offer, suppress after repeated non-response'],
 ['Needs Attention','Diagnose and nurture','Behavior-based content, preference collection, light-touch promotion'],
],columns=['Segment','Objective','RecommendedActions'])
display(strategy)
rfm.to_csv(output_dir/'rfm_customer_segments.csv',index=False)
segment_summary.to_csv(output_dir/'rfm_segment_summary.csv',index=False)
strategy.to_csv(output_dir/'rfm_segment_strategy.csv',index=False)
top=segment_summary.iloc[0]
champ=segment_summary.loc[segment_summary.Segment=='Champions'].iloc[0]
print('RFM INSIGHT SNAPSHOT')
print(f"1. Largest revenue segment: {top.Segment}, contributing {top.RevenueShare:.1%} of revenue.")
print(f"2. Champions: {int(champ.Customers):,} customers ({champ.CustomerShare:.1%}) contributing {champ.RevenueShare:.1%} of revenue.")
print('3. Every customer is assigned exactly once, and each segment has a linked lifecycle action.')